4-bit edge-feasibility benchmark for inference latency, memory use, and SQLite payload buffering.


In [ ]:
# ============================================================



# ============================================================
!pip -q uninstall -y torchao
!pip -q install -U "transformers>=4.44.0" accelerate peft datasets tqdm bitsandbytes pandas psutil

import os, json, re, time, gc, sqlite3
from datetime import datetime

import numpy as np
import pandas as pd
import psutil
import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

try:
    import bitsandbytes as bnb
    print("bitsandbytes:", bnb.__version__)
except Exception as e:
    print("bitsandbytes import failed:", repr(e))
    raise


In [ ]:
# ============================================================

# ============================================================
ADAPTER_DIR = "./qwen3_cold_chain_s3_driver_lora"
print("ADAPTER_DIR exists?", os.path.isdir(ADAPTER_DIR))
!ls -lah "$ADAPTER_DIR" | head -n 50

assert os.path.isfile(os.path.join(ADAPTER_DIR, "adapter_config.json")), "adapter_config.json not found"
assert os.path.isfile(os.path.join(ADAPTER_DIR, "adapter_model.safetensors")), "adapter_model.safetensors not found"
print("adapter files found")

cfg_path = os.path.join(ADAPTER_DIR, "adapter_config.json")
with open(cfg_path, "r", encoding="utf-8") as f:
    adapter_cfg = json.load(f)

BASE_MODEL = adapter_cfg.get("base_model_name_or_path", "Qwen/Qwen3-0.6B")
print("Using BASE_MODEL =", BASE_MODEL)


In [ ]:
# ============================================================


# ============================================================
PROMPT_TEMPLATE = """You are an assistant that generates short, practical alerts for refrigerated-truck drivers transporting strawberries.

Your job:
- Turn a structured 1-hour risk summary into a 4-line driver message.

Hard rules:
- Use this fixed format and these exact line headers:
  Line 1: Status: ...
  Line 2: Temp pattern: ...
  Line 3: Data quality: ...
  Line 4: Action: ...
- Use simple, practical language suitable for a busy driver.
  - Short sentences.
  - No technical terms (no "respiration rate", "variance", etc.).
- Focus on:
  - what is happening in this 1-hour window,
  - and what the driver should do now.
- Map risk level to Status:
  - R0 → "Status: Normal – ..."
  - R1 → "Status: Warning – ..."
  - R2 → "Status: High risk – ..."
- Map confidence level to Data quality:
  - high confidence → "Data quality: High – ..."
  - medium confidence → "Data quality: Medium – ..."
  - low confidence → "Data quality: Low – ..."
- Map the Action line as follows, and ALWAYS include a dash (–) after the main verb:
  - For R0 (normal): use "Action: Maintain – ..." with a short instruction
    (e.g., keep current settings, drive as usual).
  - For R1 (warning): use "Action: Review – ..." with a short instruction
    (e.g., reduce long door openings, check set-point and airflow).
  - For R2 (high risk): use "Action: Act now – ..." with a short instruction
    (e.g., lower the fridge set-point, keep doors closed, check cooling unit or cargo).
  - Do NOT write "Action: ... with ...". Always use the pattern "Action: <verb> – <details>".
- Use the pattern of temperatures and variation:
  - If standard deviation is low and conditions are consistent → talk about "stable" or "steady" pattern.
  - If variation is large or locations differ a lot → talk about "uneven", "mixed", or "large differences".
- Do NOT mention internal model details or equations.
- Do NOT output the summary again. Only output the 4 driver lines.
- Output EXACTLY 4 lines and NOTHING else.
- Do NOT write any labels like "Answer:", "Response:", or explanations.
- Do NOT use JSON, curly braces {{ }}, or code blocks.
- Do NOT repeat the summary.
- Do NOT invent new numbers or time durations that are not clearly in the summary.
- If you mention durations, always use minutes (e.g., "30 minutes"), never seconds.

Style rules:
- Prefer qualitative wording like "long time above 4°C" or
  "short period below 0°C" instead of exact counts of minutes.
- Only mention exact durations (e.g. "30 minutes") if that number
  is clearly written in the summary.
- Do NOT use patterns like "60 minutes above 4°C and 0 minutes below 0°C".
- For warm severe cases (R2, temperatures clearly above 4°C),
  prefer phrases like:
  - "long time above 4°C"
  - "strong warm peak"
  - "moderate temperature variation"
  - "large temperature swings"
  instead of repeating numeric statistics.

Here are examples:

Example 1 – R0, low risk, stable and in-range, high confidence
Summary:
In this 1-hour window, the assessed risk level is R0 (low). All measured temperatures stay between 0°C and 4°C with only very small variation, and there are no excursions into freezing or excessive warming. This pattern matches the recommended storage range for strawberries and should support normal shelf-life. Sensor coverage is complete and consistent, so the assessment is made with high confidence and the temperature behaviour is stable.

Driver message:
Status: Normal – safe and stable temperature.
Temp pattern: In-range – temperatures stayed between 0–4°C with only small variation.
Data quality: High – sensors give a consistent and complete cold pattern.
Action: Maintain – keep current fridge settings and drive as usual.


Example 2 – R1, moderate risk, short deviations, high confidence
Summary:
In this 1-hour window, the assessed risk level is R1 (moderate). Temperatures stay close to the 0–4°C target range but drift slightly outside it for short periods, for example briefly above 4°C or just below 0°C. These deviations are not long enough to reach severe-risk conditions, but they may cause quality loss if repeated many times. Temperature variation is moderate but not extreme, and all sensors report normally, so the assessment is made with high confidence.

Driver message:
Status: Warning – minor temperature deviation.
Temp pattern: Mixed – short time just outside 0–4°C with moderate variation.
Data quality: High – sensors show the pattern clearly.
Action: Review – reduce long door openings and confirm that set-point and airflow are correct.


Example 3 – R2, severe high temperature, long time warm, stable warm pattern, high confidence
Summary:
In this 1-hour window, the assessed risk level is R2 (severe). Temperatures are well above 4°C for almost the entire hour and there is at least one strong warm peak clearly above 10°C. The mean temperature is high and the variation is not large, so the load is consistently too warm rather than just briefly spiking. This warm exposure can strongly shorten the remaining shelf-life of strawberries. Sensor coverage is high and consistent across locations, so the assessment and the warm pattern are both considered reliable.

Driver message:
Status: High risk – cargo too warm.
Temp pattern: Over-warm – almost the whole hour well above 4°C with a strong warm peak and stable warm pattern.
Data quality: High – multiple sensors agree on this warm condition.
Action: Act now – lower the fridge set-point, keep doors closed, and check the cooling unit and airflow.


Example 4 – R2, severe high temperature, uneven and with missing data, medium confidence
Summary:
In this 1-hour window, the assessed risk level is R2 (severe). Temperatures are above 4°C for a long time and there is a strong warm peak above 10°C, indicating that at least part of the load is over-warm. However, some sensor readings are missing or incomplete, and the variation between locations is quite large. This means parts of the cargo may be much warmer than others, but the exact distribution is uncertain. Because of the missing data and uneven pattern, the assessment is made with only medium confidence.

Driver message:
Status: High risk – cargo likely too warm.
Temp pattern: Over-warm and uneven – long time above 4°C with a strong warm peak and large differences inside the load.
Data quality: Medium – some sensor readings are missing or incomplete.
Action: Act now – lower the set-point, keep doors closed, and, if possible, check the cargo condition manually.


Example 5 – R2, severe risk, warm for too long without extreme peak, high confidence
Summary:
In this 1-hour window, the assessed risk level is R2 (severe). Temperatures stay above 4°C for around half an hour, while the rest of the period is close to the lower limit of the recommended range. Even without an extreme peak above 10°C, this prolonged moderate warming can still accelerate quality loss, especially if similar episodes repeat during the trip. Temperature variation is moderate and sensor readings are complete across time and locations, so the warm period is clearly visible and the assessment is made with high confidence.

Driver message:
Status: High risk – cargo warm for too long.
Temp pattern: Over-warm – about half an hour above 4°C and the rest near the lower limit, with moderate variation.
Data quality: High – sensors show a clear warm period.
Action: Act now – shorten door-opening times, check fridge capacity, and consider a slightly lower set-point.


Example 6 – R2, severe risk, long period near/below 0°C, high confidence
Summary:
In this 1-hour window, the assessed risk level is R2 (severe). The maximum temperature stays at or below 4°C, but minimum values drop close to or slightly below 0°C, and the cargo spends around 30 minutes at or below 0°C. This over-cooling near the freezing point can cause chilling injury or partial freezing if it occurs repeatedly. The temperature pattern is fairly stable and cold, and all sensors report consistently, so the assessment is made with high confidence.

Driver message:
Status: High risk – cargo too cold.
Temp pattern: Over-cold – around 30 minutes close to or below 0°C with a stable cold pattern.
Data quality: High – sensors show a consistent cold trend.
Action: Act now – raise the set-point slightly, avoid very strong cooling, and watch the coldest positions.


Example 7 – R2, severe risk, possible freezing damage, high confidence
Summary:
In this 1-hour window, the assessed risk level is R2 (severe). Temperatures are below 0°C for most of the hour, and the coldest locations drop to around or below −1°C. This cold exposure is close to or beyond the freezing point of strawberries and can cause irreversible freezing damage, such as darkening, watery texture, and collapse after re-warming. The cold pattern is steady and several sensors agree on this behaviour, so the assessment is made with high confidence.

Driver message:
Status: High risk – possible freezing damage.
Temp pattern: Freezing – long time below 0°C with the coldest spots near or below −1°C and a steady cold pattern.
Data quality: High – several sensors confirm this freezing behaviour.
Action: Act now – increase the set-point, avoid strong cooling, and flag this batch for a quality check at destination.

----------------
Now a new case:
Summary:
{PUT_SUMMARY_TEXT_HERE}

Please generate ONLY the 4-line driver message in the required format:
Line 1: Status: ...
Line 2: Temp pattern: ...
Line 3: Data quality: ...
Line 4: Action: ...
"""

In [ ]:
# ============================================================




# ============================================================
def clean_driver_message(raw_text: str) -> str:
    required_prefixes = ["Status:", "Temp pattern:", "Data quality:", "Action:"]
    collected = {p: None for p in required_prefixes}

    for line in raw_text.splitlines():
        line = line.strip()
        if not line:
            continue

        for p in required_prefixes:
            if collected[p] is None and line.startswith(p):
                collected[p] = line
                break

        if all(collected[p] is not None for p in required_prefixes):
            break

    if all(collected[p] is not None for p in required_prefixes):
        return "\n".join([collected[p] for p in required_prefixes])
    else:
        return raw_text.strip()


_FMT_OK_RE = re.compile(
    r"(?is)Status:\s*(.*?)\s*Temp pattern:\s*(.*?)\s*Data quality:\s*(.*?)\s*Action:\s*(.*)"
)

def is_format_ok(msg: str) -> bool:
    return isinstance(msg, str) and bool(_FMT_OK_RE.search(msg))


In [ ]:
# ============================================================


# ============================================================

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()

cleanup_cuda()

qconfig = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tok_4bit = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False, trust_remote_code=True)
if tok_4bit.pad_token_id is None:
    tok_4bit.pad_token_id = tok_4bit.eos_token_id

base_4bit = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=qconfig,
    device_map="auto",
    trust_remote_code=True,
)
base_4bit.config.pad_token_id = tok_4bit.pad_token_id

m_4bit = PeftModel.from_pretrained(base_4bit, ADAPTER_DIR)
m_4bit.eval()

print(" 4-bit model ready")
if torch.cuda.is_available():
    print("Current allocated after loading (GB):", round(torch.cuda.memory_allocated() / (1024**3), 3))
    print("Max allocated after loading (GB):", round(torch.cuda.max_memory_allocated() / (1024**3), 3))


In [ ]:
# ============================================================

# ============================================================
MAX_PROMPT_TOKENS = 2048

def make_runner(model, tokenizer, prompt_template: str, max_prompt_tokens: int = 2048):
    fmt_re = re.compile(r"(?is)Status:.*?Temp pattern:.*?Data quality:.*?Action:")

    def encode_left_truncate(prompt_for_model: str):
        enc = tokenizer(prompt_for_model, add_special_tokens=False, return_tensors="pt")
        input_ids = enc["input_ids"][0]
        if input_ids.numel() > max_prompt_tokens:
            input_ids = input_ids[-max_prompt_tokens:]
        attn = torch.ones_like(input_ids)
        return {
            "input_ids": input_ids.unsqueeze(0).to(model.device),
            "attention_mask": attn.unsqueeze(0).to(model.device),
        }

    @torch.no_grad()
    def generate_once(summary: str, max_new_tokens: int = 160, min_new_tokens: int = 32, debug: bool = False):
        prompt = prompt_template.format(PUT_SUMMARY_TEXT_HERE=str(summary))
        prompt_for_model = prompt + "\n"
        inputs = encode_left_truncate(prompt_for_model)
        input_len = int(inputs["input_ids"].shape[1])

        out = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )

        gen_ids = out[0][input_len:]
        raw = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        cleaned = clean_driver_message(raw)

        meta = {
            "input_len": input_len,
            "gen_tokens": int(gen_ids.numel()),
            "raw_len_chars": len(raw),
            "format_ok": bool(fmt_re.search(cleaned)),
        }

        if debug:
            print(f"[debug] input_len={meta['input_len']} gen_tokens={meta['gen_tokens']} format_ok={meta['format_ok']}")
            print("[debug] raw head:\n", raw[:400])

        return cleaned, meta

    def run(summary: str, max_new_tokens: int = 256, retries: int = 1, debug: bool = False):
        last_msg, last_meta = None, None
        for attempt in range(retries + 1):
            msg, meta = generate_once(summary, max_new_tokens=max_new_tokens, min_new_tokens=32, debug=debug)
            meta.update({"attempt": attempt, "retried": attempt > 0})
            last_msg, last_meta = msg, meta
            if meta["format_ok"]:
                return msg, meta
        return last_msg, last_meta

    return run

run_4bit = make_runner(m_4bit, tok_4bit, PROMPT_TEMPLATE, max_prompt_tokens=MAX_PROMPT_TOKENS)
print(" 4-bit runner ready")


In [ ]:
# ============================================================


# ============================================================
dataset_name = "NifferLi/cold-chain-strawberry-sensors"
ds = load_dataset(dataset_name)

SPLIT_FOR_BENCH = "s4"
N_BENCH = 1000

bench_ds = ds[SPLIT_FOR_BENCH].filter(lambda x: bool(str(x.get("summary_text", "")).strip()))
N_BENCH = min(N_BENCH, len(bench_ds))

rng = np.random.RandomState(0)
idx = rng.choice(len(bench_ds), size=N_BENCH, replace=False)
bench_summaries = [bench_ds[int(i)]["summary_text"] for i in idx]

print("Bench split:", SPLIT_FOR_BENCH, "| N_BENCH:", len(bench_summaries))
print("Example head:\n", bench_summaries[0][:300])


In [ ]:
# ============================================================


# ============================================================
def fmt_seconds(sec: float) -> str:
    sec = int(round(sec))
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    if h > 0:
        return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"

def bytes_to_gb(x: int) -> float:
    return float(x) / (1024**3)

LATENCY_TARGET_MS = 60_000.0
MONITORING_CYCLE_MS = 10 * 60 * 1000.0

def benchmark_inference(run_fn, summaries, warmup=5, desc="4-bit benchmark"):
    proc = psutil.Process(os.getpid())

    for i in range(min(warmup, len(summaries))):
        _ = run_fn(summaries[i], max_new_tokens=160, retries=1, debug=False)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    lat_ms = []
    messages = []
    format_ok_list = []
    gen_tokens_list = []
    total_gen_tokens = 0
    total_time_s = 0.0
    peak_rss = 0
    ok_cnt = 0

    pbar = tqdm(summaries, desc=desc, unit="sample")
    t_start = time.perf_counter()

    for k, s in enumerate(pbar, start=1):
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        t0 = time.perf_counter()
        msg, meta = run_fn(s, max_new_tokens=160, retries=1, debug=False)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        dt = t1 - t0
        lat_ms.append(dt * 1000.0)
        messages.append(msg)
        is_ok = bool(meta.get("format_ok") is True)
        format_ok_list.append(is_ok)
        gen_tokens = int(meta.get("gen_tokens") or 0)
        gen_tokens_list.append(gen_tokens)

        total_time_s += dt
        total_gen_tokens += gen_tokens
        ok_cnt += int(is_ok)
        peak_rss = max(peak_rss, proc.memory_info().rss)

        if k % 20 == 0:
            p50 = float(np.percentile(lat_ms, 50))
            pmax = float(np.max(lat_ms))
            ok_rate = ok_cnt / k
            elapsed = time.perf_counter() - t_start
            pbar.set_postfix({
                "p50(ms)": round(p50, 1),
                "max(ms)": round(pmax, 1),
                "ok%": round(ok_rate*100, 1),
                "elapsed": fmt_seconds(elapsed),
            })

    peak_vram = None
    if torch.cuda.is_available():
        peak_vram = int(torch.cuda.max_memory_allocated())

    lat_arr = np.array(lat_ms, dtype=float)

    return {
        "n": len(lat_ms),
        "lat_p50_ms": float(np.percentile(lat_arr, 50)),
        "lat_p95_ms": float(np.percentile(lat_arr, 95)),
        "lat_p99_ms": float(np.percentile(lat_arr, 99)),
        "lat_max_ms": float(np.max(lat_arr)),
        "exceed_60s_pct": float(np.mean(lat_arr > LATENCY_TARGET_MS) * 100.0),
        "exceed_10min_pct": float(np.mean(lat_arr > MONITORING_CYCLE_MS) * 100.0),
        "throughput_tokens_per_s": (total_gen_tokens / total_time_s) if total_time_s > 0 else None,
        "peak_ram_bytes": int(peak_rss),
        "peak_vram_bytes": int(peak_vram) if peak_vram is not None else None,
        "format_ok_rate": ok_cnt / len(lat_ms) if len(lat_ms) else None,
        "total_time_s": total_time_s,
        "total_gen_tokens": total_gen_tokens,
        "latencies_ms": lat_ms,
        "messages": messages,
        "format_ok_list": format_ok_list,
        "gen_tokens_list": gen_tokens_list,
    }


In [ ]:
# ============================================================



# ============================================================
import sqlite3

def benchmark_sqlite_buffer(payloads, db_path="outputs/edge_buffer.db"):
    if os.path.exists(db_path):
        os.remove(db_path)

    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("""
    CREATE TABLE IF NOT EXISTS alerts(
      id INTEGER PRIMARY KEY AUTOINCREMENT,
      ts REAL,
      payload TEXT,
      uploaded INTEGER DEFAULT 0
    )
    """)
    conn.commit()

    ts = time.time()


    t0 = time.perf_counter()
    cur.executemany("INSERT INTO alerts(ts,payload,uploaded) VALUES(?,?,0)", [(ts, p) for p in payloads])
    conn.commit()
    t1 = time.perf_counter()
    write_s = (t1 - t0)


    t2 = time.perf_counter()
    cur.execute("SELECT id FROM alerts WHERE uploaded=0")
    ids = [r[0] for r in cur.fetchall()]
    cur.executemany("UPDATE alerts SET uploaded=1 WHERE id=?", [(i,) for i in ids])
    conn.commit()
    t3 = time.perf_counter()
    flush_s = (t3 - t2)

    conn.close()

    n = len(payloads)
    return {
        "buffer_n": n,
        "sqlite_write_ms_total": write_s * 1000.0,
        "sqlite_write_ms_per_record": (write_s * 1000.0 / n) if n > 0 else None,
        "buffer_flush_ms": flush_s * 1000.0,
        "flush_records_per_s": (n / flush_s) if flush_s > 0 else None,
    }


In [ ]:
# ============================================================


# ============================================================
ARTIFACT_ROOT = "outputs/edge_feasibility_outputs_reruns_4bit_only"
RUN_TAG = datetime.now().strftime("rerun_%Y%m%d_%H%M%S")
ARTIFACT_DIR = os.path.join(ARTIFACT_ROOT, RUN_TAG)
os.makedirs(ARTIFACT_DIR, exist_ok=True)
print("ARTIFACT_DIR:", ARTIFACT_DIR)

print("==== [1/2] 4bit inference benchmark ====")
r_4bit = benchmark_inference(run_4bit, bench_summaries, warmup=5, desc="4bit benchmark")
metrics_4bit = {k: v for k, v in r_4bit.items() if k not in ["latencies_ms", "messages", "format_ok_list", "gen_tokens_list"]}
print(metrics_4bit, "\n")

# Save inference metrics JSON
metrics_path = os.path.join(ARTIFACT_DIR, "edge_metrics_4bit_inference.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics_4bit, f, ensure_ascii=False, indent=2)
print(" Saved:", metrics_path)

# Save latency trace CSV
trace_df = pd.DataFrame({
    "idx": list(range(len(r_4bit["latencies_ms"]))),
    "latency_ms": r_4bit["latencies_ms"],
    "format_ok": r_4bit["format_ok_list"],
    "gen_tokens": r_4bit["gen_tokens_list"],
})
trace_path = os.path.join(ARTIFACT_DIR, "edge_latency_trace_4bit.csv")
trace_df.to_csv(trace_path, index=False)
print(" Saved 4-bit latency trace:", trace_path)

# Save generated payloads/messages CSV
payload_df = pd.DataFrame({
    "idx": list(range(len(r_4bit["messages"]))),
    "driver_message_pred": r_4bit["messages"],
    "format_ok": r_4bit["format_ok_list"],
})
payload_path = os.path.join(ARTIFACT_DIR, "edge_payloads_4bit.csv")
payload_df.to_csv(payload_path, index=False)
print(" Saved 4-bit payloads:", payload_path)

print("==== [2/2] 4bit payloads + sqlite ====")
N_BUFFER_PAYLOAD = min(1000, len(r_4bit["messages"]))
payloads_4bit = r_4bit["messages"][:N_BUFFER_PAYLOAD]
b_4bit = benchmark_sqlite_buffer(payloads_4bit, db_path="outputs/edge_buffer_4bit_only.db")
print(b_4bit, "\n")

sqlite_path = os.path.join(ARTIFACT_DIR, "edge_metrics_4bit_sqlite.json")
with open(sqlite_path, "w", encoding="utf-8") as f:
    json.dump(b_4bit, f, ensure_ascii=False, indent=2)
print(" Saved:", sqlite_path)

# Final one-row table for easy copy/check
row_4bit = {
    "Precision": "4bit",
    "Latency p50 (ms)": round(r_4bit["lat_p50_ms"], 1),
    "Latency p95 (ms)": round(r_4bit["lat_p95_ms"], 1),
    "Latency p99 (ms)": round(r_4bit["lat_p99_ms"], 1),
    "Latency max (ms)": round(r_4bit["lat_max_ms"], 1),
    "Exceed 60s pct": round(r_4bit["exceed_60s_pct"], 3),
    "Exceed 10min pct": round(r_4bit["exceed_10min_pct"], 3),
    "Throughput (tokens/s)": round(r_4bit["throughput_tokens_per_s"], 2),
    "Peak RAM (GB)": round(bytes_to_gb(r_4bit["peak_ram_bytes"]), 3),
    "Peak VRAM (GB)": round(bytes_to_gb(r_4bit["peak_vram_bytes"]), 3) if r_4bit["peak_vram_bytes"] is not None else None,
    "SQLite write (ms) [N=1000]": round(b_4bit["sqlite_write_ms_total"], 2),
    "Write (ms/record)": round(b_4bit["sqlite_write_ms_per_record"], 4),
    "Buffer flush (ms) [N=1000]": round(b_4bit["buffer_flush_ms"], 2),
    "Flush throughput (records/s)": round(b_4bit["flush_records_per_s"], 1),
    "4-line OK rate": round(r_4bit["format_ok_rate"], 3),
}

df_4bit = pd.DataFrame([row_4bit])
display(df_4bit)

final_csv = os.path.join(ARTIFACT_DIR, "edge_feasibility_4bit_only.csv")
df_4bit.to_csv(final_csv, index=False)
print(" Saved final 4-bit table:", final_csv)
print(" DONE. Use this 4-bit row together with the clean FP16 row from a separate fresh-runtime run.")
